# Grammar Scoring Engine for Spoken Audio — SHL Hiring Assessment

**Task.** Predict a continuous grammar score (0–5, MOS-style Likert) for each 45–60 s speech recording. Leaderboard metrics: Pearson correlation and RMSE.

**Approach in one paragraph.** Whisper (run locally, once) turns each recording into a transcript with timestamps. From the transcript and timestamps we build small, explainable feature groups (text statistics, GPT-2 perplexity, optional LanguageTool errors, fluency/pauses) and dense embeddings (sentence encoder, RoBERTa, WavLM audio). Ridge / SVR models on embeddings and LightGBM / Ridge on engineered features are evaluated with **repeated stratified 5-fold CV**; an ablation table shows what each feature group and model contributes; the best three models are combined by a **non-negative linear stack** fitted on out-of-fold predictions; predictions are clipped to [0, 5].

**Pipeline:** `00_baseline` (quick submission) → `01_extract_features` (all expensive work, cached) → **this notebook** (loads caches only; Whisper is never run here).

**Rules respected.** No external datasets · all models run locally on Kaggle (no data sent to any API) · open-source libraries only · no test labels used (test labels are random placeholders and are never used) · every imputer/scaler/penalty is fitted inside CV folds.

> ⚠️ **Honesty note.** This code was written without access to the data or a GPU. No score in this notebook is pre-filled: every number appears only when you run the cells. See the README for the list of things that were not verified.

In [ ]:
# =====================  CONFIG  -- the ONLY cell you should need to edit  =====================
import sys
from pathlib import Path

CONFIG = dict(
    # ---- where the data is (Kaggle mounts it read-only; nothing is uploaded anywhere) ----
    DATA_DIR        = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final",
    TRAIN_CSV       = None,    # None -> auto-discover train.csv under DATA_DIR
    TEST_CSV        = None,
    SAMPLE_SUB_CSV  = None,
    TRAIN_AUDIO_DIR = None,    # None -> auto-discover (folder with 'train' in its name that holds audio)
    TEST_AUDIO_DIR  = None,
    ID_COL          = None,    # None -> auto-detect the file-name column
    LABEL_COL       = None,    # None -> auto-detect the numeric label column
    # ---- where caches / results go ----
    CACHE_DIR       = "/kaggle/working/cache",
    OUTPUT_DIR      = "/kaggle/working",
    # ---- models: hub name (Internet ON)  OR  local / Kaggle-Models path (Internet OFF, set OFFLINE=True) ----
    OFFLINE         = False,
    MODEL_CACHE_DIR = None,
    WHISPER_MODEL   = "small",      # faster-whisper size, or path to a CTranslate2 Whisper folder
    BASELINE_TEXT_ENCODER = "sentence-transformers/all-MiniLM-L6-v2",
    TEXT_ENCODER    = "sentence-transformers/all-mpnet-base-v2",
    HF_ENCODER      = "roberta-base",
    LM_NAME         = "gpt2",
    AUDIO_ENCODER   = "microsoft/wavlm-base-plus",
    LLM_NAME        = "Qwen/Qwen2.5-1.5B-Instruct",
    # ---- switches ----
    RUN_LANGUAGETOOL = True,        # auto-skipped if Java / download is unavailable
    RUN_LLM_RUBRIC   = False,       # optional zero-shot rubric feature
    FORCE_RECOMPUTE  = False,       # True -> ignore cached feature files
    # ---- reproducibility / validation ----
    SEED = 42, CV_SPLITS = 5, CV_REPEATS = 3,
)

def _find_repo():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/shl-grammar-scoring")]
    for pat in ("*", "*/*", "*/*/*", "*/*/*/*"):            # Kaggle nests datasets 1-4 levels deep
        cands += list(Path("/kaggle/input").glob(pat))
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    raise FileNotFoundError("Cannot find the repo's src/ folder. Add the repo as a Kaggle Dataset (or clone it) and set REPO_DIR by hand.")

REPO_DIR = _find_repo()
sys.path.insert(0, str(REPO_DIR))
from src.config import Config
cfg = Config(**CONFIG).apply_env().seed_everything()      # env vars FIRST, before torch/transformers are imported
print("repo:", REPO_DIR)
print(cfg.summary())

## 0 · Load the cached features

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

from src import plots
from src.data import discover_layout
from src.metrics import rmse, pearson, clip_scores
from src.models import (Experiment, make_constant, make_ridge, make_svr, make_lgbm, stratify_labels,
                        run_experiments, fit_stack, stack_cv)
from src.submission import write_submission

fd = cfg.features_dir
meta = pd.read_csv(fd / "meta.csv")
F = pd.read_csv(fd / "features.csv")
ainfo = pd.read_csv(fd / "audio_info.csv")
tx = pd.read_csv(cfg.transcripts_csv)
tx["text"] = tx["text"].fillna("").astype(str)
meta = meta.merge(tx[["split", "stem", "text"]], on=["split", "stem"], how="left")
meta["text"] = meta["text"].fillna("")
E = {k: np.load(fd / f"{k}.npy") for k in ("text_st", "text_hf", "audio_wavlm") if (fd / f"{k}.npy").exists()}

tr = (meta.split == "train").to_numpy(); te = ~tr
y = meta.loc[tr, "label"].to_numpy(float)
assert len(meta) == len(F) and all(len(v) == len(meta) for v in E.values()), "cache files are misaligned: re-run 01"
print(f"train rows: {tr.sum()} | test rows: {te.sum()} | engineered features: {F.shape[1]} | embeddings: { {k: v.shape[1] for k, v in E.items()} }")
OUT = cfg.output_dir

## 1 · Exploratory data analysis
Label distribution, how far a "learned nothing" model is from perfect, audio durations, sample rates, and any missing/corrupt files.

In [ ]:
fig = plots.plot_label_distribution(y); plots.save(fig, OUT, "label_distribution"); plt.show()
print(pd.Series(y).describe().round(3).to_string())
print("\nlabel counts (rounded):\n", pd.Series(np.floor(y + 0.5).astype(int)).value_counts().sort_index().to_string())
print(f"\nCONSTANT BASELINE: always predicting the training mean gives RMSE = label std = {y.std():.4f}  (Pearson undefined)")

In [ ]:
a = ainfo.copy()
print("sample rates (Hz) in the ORIGINAL files:\n", a.samplerate.value_counts(dropna=False).to_string())
print("\nchannels:\n", a.channels.value_counts(dropna=False).to_string())
bad = a[~a.ok]
print(f"\nmissing / unreadable files: {len(bad)}")
if len(bad): display(bad[["split", "stem", "error"]].head(20))
for sp in ("train", "test"):
    d = a[(a.split == sp) & a.ok].duration
    print(f"{sp}: duration min/median/max = {d.min():.1f} / {d.median():.1f} / {d.max():.1f} s  (n={len(d)})")
fig = plots.plot_hist(a[a.ok].duration, "Original audio duration", "seconds"); plots.save(fig, OUT, "duration"); plt.show()
fig = plots.plot_hist(F.loc[:, "hand_n_words"], "Words per transcript", "words"); plots.save(fig, OUT, "words_per_transcript"); plt.show()
print("empty transcripts:", int((meta.text.str.len() == 0).sum()))

## 2 · Preprocessing (done in `01`; parameters shown for the record)
1. `librosa.load(..., sr=16000, mono=True)` — resample to 16 kHz and mix down to mono (Whisper and WavLM both expect 16 kHz).
2. Trim **leading/trailing** silence (`top_db` below) — internal pauses are kept because they carry fluency information.
3. Normalise loudness: scale to a fixed RMS, then limit the peak to 0.99 — removes recording-level differences.
4. Cache as 16-bit wav; files that cannot be read get an empty transcript instead of crashing the run.

In [ ]:
print(f"sample rate = {cfg.SAMPLE_RATE} Hz | trim top_db = {cfg.TRIM_TOP_DB} | target RMS = {cfg.TARGET_RMS}")

## 3 · Transcription (done in `01`/`00`; examples)
Whisper (`faster-whisper`, model set in CONFIG) transcribes locally; forced to English, beam search, no VAD (so real pauses stay visible), word-level timestamps and confidences stored.  
Known behaviour: Whisper tends to **clean up** disfluencies and sometimes corrects grammar, which hides some of the signal we want (see Limitations).

In [ ]:
for _, r in meta[tr].sample(4, random_state=cfg.SEED).iterrows():
    # transcript text is deliberately NOT printed (derived from competition data; must not be published)
    print(f"[label {r.label}] {r.stem} | words: {len(r.text.split())}")

## 4 · Feature engineering
Feature groups (column prefix → meaning):
* `hand_` text statistics (length, vocabulary variety, sentence shape, fillers, repetitions, false starts, subordinators)
* `lm_` GPT-2 token loss / perplexity
* `lt_` LanguageTool error counts (only if it ran)
* `flu_` speech rate, pauses, ASR confidence, duration
* `llm_` optional zero-shot LLM rubric score (only if it ran)
* embeddings: `text_st` (sentence encoder), `text_hf` (RoBERTa mean-pool), `audio_wavlm` (WavLM mean+std)

In [ ]:
ORDER = [g for g in ("hand", "lm", "lt", "flu", "llm") if any(c.startswith(g + "_") for c in F.columns)]
groups = {}
for g in ORDER:
    cols = [c for c in F.columns if c.startswith(g + "_")]
    # drop columns that are all-missing or constant in TRAIN (they carry no information; avoids imputer shape surprises)
    groups[g] = [c for c in cols if F.loc[tr, c].notna().any() and F.loc[tr, c].nunique() > 1]
print({g: len(c) for g, c in groups.items()})
print("groups NOT available:", [g for g in ("hand", "lm", "lt", "flu", "llm") if g not in groups])
eng_cols = [c for g in ORDER for c in groups[g]]
print("missing values per group (train+test rows):", {g: int(F[c].isna().sum().sum()) for g, c in groups.items()})

fig = plots.plot_feature_corr(F.loc[tr, eng_cols].reset_index(drop=True), y, top=25)
plots.save(fig, OUT, "feature_correlation"); plt.show()

## 5 · Modeling and cross-validation

**Validation protocol.** Repeated 5-fold CV, stratified on the rounded label (classes smaller than the fold count are merged into their neighbour), fixed seeds. For **every fold** the RMSE and Pearson are printed, then the overall out-of-fold values. Predictions are clipped to [0, 5].

**Models.**
* *Ridge* — a linear model with an L2 penalty; the penalty strength is chosen by leave-one-out CV **inside the training part of each fold**. Good for embeddings (many correlated dims, few rows).
* *SVR (RBF)* — a smooth non-linear regressor on embeddings, fixed conservative settings.
* *LightGBM* — small, shallow, regularised trees for the engineered tabular features (captures thresholds and interactions; handles missing values).
* *Constant mean* — the "learned nothing" reference.

Imputation (median) and standardisation sit inside each model's `Pipeline`, so they are fitted on training rows only.

In [ ]:
SEEDS = [cfg.SEED + i for i in range(cfg.CV_REPEATS)]
strat = stratify_labels(y, cfg.CV_SPLITS)
print("strata sizes:", dict(zip(*np.unique(strat, return_counts=True))), "| CV seeds:", SEEDS)

exps = []
def add(name, feats, model, factory, A):
    exps.append(Experiment(name, feats, model, factory, A[tr], A[te]))
lgbm = lambda: make_lgbm(cfg.SEED)
tab = lambda cols: F[cols].to_numpy(float)

add("constant_mean", "(none)", "constant", make_constant, np.zeros((len(F), 1)))

# (a) each engineered feature group on its own: Ridge and LightGBM
for g in ORDER:
    add(f"ridge_{g}", g, "ridge", make_ridge, tab(groups[g]))
    add(f"lgbm_{g}", g, "lgbm", lgbm, tab(groups[g]))

# (b) cumulative LightGBM: what does each added group give on top of the previous ones?
for i in range(2, len(ORDER) + 1):
    cols = [c for g in ORDER[:i] for c in groups[g]]
    nm = "lgbm_eng" if i == len(ORDER) else "lgbm_cum_" + "+".join(ORDER[:i])
    add(nm, "+".join(ORDER[:i]), "lgbm", lgbm, tab(cols))
add("ridge_eng", "+".join(ORDER), "ridge", make_ridge, tab(eng_cols))

# (c) each embedding on its own: Ridge and SVR
for k, A in E.items():
    add(f"ridge_{k}", k, "ridge", make_ridge, A)
    add(f"svr_{k}", k, "svr", make_svr, A)

# (d) combinations
if "text_st" in E and "text_hf" in E:
    add("ridge_text_emb", "text_st+text_hf", "ridge", make_ridge, np.hstack([E["text_st"], E["text_hf"]]))
all_emb = np.hstack(list(E.values()))
add("ridge_all_emb", "+".join(E), "ridge", make_ridge, all_emb)
add("ridge_everything", "all embeddings + engineered", "ridge", make_ridge, np.hstack([all_emb, tab(eng_cols)]))
print(len(exps), "experiments:", [e.name for e in exps])

In [ ]:
VERBOSE_FOLDS = True      # prints RMSE and Pearson for every fold of every experiment (set False for a quieter log)
results, cv = run_experiments(exps, y, strat, SEEDS, cfg.CV_SPLITS, verbose=VERBOSE_FOLDS)
results.to_csv(OUT / "ablation_results.csv", index=False)

## 6 · CV results and ablation
`rmse_overall` / `pearson_overall` = metric of the full out-of-fold vector, averaged over repeats. `*_fold_mean ± std` = spread across individual folds (shows how noisy 770 samples are).

In [ ]:
const_rmse = cv["constant_mean"]["summary"]["rmse_overall"]
cols = ["name", "features", "model", "n_features", "rmse_overall", "pearson_overall",
        "rmse_fold_mean", "rmse_fold_std", "pearson_fold_mean", "pearson_fold_std"]
print(f"label std = {y.std():.4f} | constant-mean CV RMSE = {const_rmse:.4f}")
display(results[cols].round(4))
fig = plots.plot_ablation(results, const_rmse); plots.save(fig, OUT, "ablation"); plt.show()

In [ ]:
ranked = results[results.name != "constant_mean"].reset_index(drop=True)
best = ranked.iloc[0]
print(f"Best single experiment: {best['name']}  | CV RMSE {best.rmse_overall:.4f} | CV Pearson {best.pearson_overall:.4f}")
oof_best = cv[best["name"]]["oof"]
for f, nm in ((plots.plot_oof_scatter(y, oof_best, best["name"]), "oof_scatter"),
              (plots.plot_residual_hist(y, oof_best), "residuals"),
              (plots.plot_per_score_error(y, oof_best), "per_score_error")):
    plots.save(f, OUT, nm); plt.show()

## 7 · Final ensemble (out-of-fold weights)
1. Take the **top 3** experiments by CV RMSE.  
2. Learn a **non-negative linear stack** (weights ≥ 0 plus an intercept) on their out-of-fold predictions.  
3. Because those weights are fitted on the same OOF predictions they are scored on, we also run a **second-level CV** (weights fitted on part of the OOF matrix, scored on the rest) to get an honest estimate.  
4. We compare: best single model, plain average of the 3, and the stack — and use whichever has the lowest honest CV RMSE.

Caveat: choosing the top 3 on the same CV used to score them is a mild optimistic selection effect; with ~770 rows differences of ~0.01 RMSE are within noise.

In [ ]:
cands = ranked["name"].head(3).tolist()
P = np.column_stack([cv[n]["oof"] for n in cands])                    # OOF prediction matrix (n_train, 3)
R = len(SEEDS)
def blend_rmse(wfun):                                                  # mean over repeats of the full-OOF RMSE
    return float(np.mean([rmse(y, wfun(np.column_stack([cv[n]["oof_repeats"][r] for n in cands]))) for r in range(R)]))

stack = fit_stack(P, y)
print("candidates:", cands)
print("stack weights:", dict(zip(cands, stack.coef_.round(3).tolist())), "| intercept:", round(float(stack.intercept_), 3))
st_cv = stack_cv(P, y, strat, [s + 1000 for s in SEEDS], cfg.CV_SPLITS, verbose=True)

options = {
    "best_single": dict(rmse=float(best.rmse_overall), w=np.array([1.0, 0, 0]), b=0.0),
    "mean_of_3":   dict(rmse=blend_rmse(lambda M: M.mean(1)), w=np.ones(3) / 3, b=0.0),
    "stack":       dict(rmse=st_cv["rmse_overall"], w=stack.coef_.copy(), b=float(stack.intercept_)),
}
print({k: round(v["rmse"], 4) for k, v in options.items()})
chosen = min(options, key=lambda k: options[k]["rmse"])
W, B = options[chosen]["w"], options[chosen]["b"]
print("CHOSEN ensemble:", chosen)
oof_ens = clip_scores(P @ W + B)
cv_ens_rmse = options[chosen]["rmse"]
cv_ens_pearson = (st_cv["pearson_overall"] if chosen == "stack" else
                  float(np.nanmean([pearson(y, clip_scores(np.column_stack([cv[n]["oof_repeats"][r] for n in cands]) @ W + B)) for r in range(R)])))

## 8 · Final fit on all training data and test prediction
Each of the selected models is re-fitted on **all** training rows (imputer, scaler and ridge penalty are fitted on these rows only), then the ensemble weights chosen above are applied. Test rows are only ever passed to `predict`.

In [ ]:
exp_by_name = {e.name: e for e in exps}
train_preds, test_preds, fitted = [], [], {}
for n in cands:
    e = exp_by_name[n]
    m = e.factory().fit(e.X_train, y)
    fitted[n] = m
    train_preds.append(clip_scores(m.predict(e.X_train)))
    test_preds.append(clip_scores(m.predict(e.X_test)))
Ptr, Pte = np.column_stack(train_preds), np.column_stack(test_preds)
train_pred = clip_scores(Ptr @ W + B)
test_pred = clip_scores(Pte @ W + B)
train_rmse = rmse(y, train_pred)

print("=" * 76)
print("                           FINAL MODEL SUMMARY")
print("=" * 76)
print(f"TRAINING RMSE  (final ensemble, fit AND scored on the full training set): {train_rmse:.4f}")
print(f"CV RMSE        (out-of-fold, honest)                                    : {cv_ens_rmse:.4f}")
print(f"CV Pearson     (out-of-fold, honest)                                    : {cv_ens_pearson:.4f}")
print(f"Constant baseline RMSE (label std, 'learned nothing')                   : {y.std():.4f}")
print("=" * 76)
print("Training RMSE is optimistic (the model has seen these labels). A large gap to CV RMSE means over-fitting.")
print({n: round(rmse(y, p), 4) for n, p in zip(cands, train_preds)}, "<- training RMSE of each base model")

In [ ]:
# The grader expects 216 rows = test.csv (sample_submission.csv has only 204, with other names).
sample_csv = cfg.TEST_CSV or discover_layout(cfg.DATA_DIR)["test_csv"]
sub = write_submission(test_pred, meta[te].reset_index(drop=True), sample_csv, str(OUT / "submission.csv"))
display(sub.head())
print(sub.iloc[:, -1].describe().round(3).to_string())

In [ ]:
# Feature importance / coefficients (engineered features), refitted on all training rows just for explanation
ridge_eng = make_ridge().fit(tab(eng_cols)[tr], y)
coef = ridge_eng.named_steps["m"].coef_                       # on standardised features -> comparable magnitudes
fig = plots.plot_importance(eng_cols, coef, "Ridge coefficients (standardised engineered features)", top=20)
plots.save(fig, OUT, "ridge_coefficients"); plt.show()
print("chosen ridge alpha:", ridge_eng.named_steps["m"].alpha_)

lg = make_lgbm(cfg.SEED).fit(tab(eng_cols)[tr], y)
if hasattr(lg, "booster_"):
    gain = lg.booster_.feature_importance(importance_type="gain")
    fig = plots.plot_importance(eng_cols, gain / gain.sum(), "LightGBM importance (share of total gain)", top=20)
    plots.save(fig, OUT, "lgbm_importance"); plt.show()
else:
    print("LightGBM not installed -> sklearn fallback used; no gain importance available.")

## 9 · Interpretability: why did the model score these files high or low?
Below: the training files with the **highest and lowest out-of-fold ensemble scores** (these are training files whose labels we know; test files are never inspected). For each, the engineered features that pushed the (Ridge) score up or down the most: `contribution = coefficient × standardised value`, relative to an average file.  
This explains the **engineered-feature view** of the model; embeddings are not individually interpretable.

In [ ]:
imp, sc, rid = (ridge_eng.named_steps[k] for k in ("imp", "sc", "m"))
Xe = tab(eng_cols)[tr]
Z = sc.transform(imp.transform(Xe))
train_text = meta.loc[tr, "text"].to_numpy()
train_stem = meta.loc[tr, "stem"].to_numpy()
order_idx = np.argsort(oof_ens)

def explain(i, k=5):
    contrib = rid.coef_ * Z[i]
    top = np.argsort(-np.abs(contrib))[:k]
    print(f"{train_stem[i]} | true {y[i]:.2f} | OOF ensemble prediction {oof_ens[i]:.2f}")
    print("  transcript length:", len(train_text[i].split()), "words (text not printed: competition data)")
    for j in top:
        print(f"   {'↑' if contrib[j] > 0 else '↓'} {eng_cols[j]:28s} value={Xe[i, j]:9.3f}  (train mean {np.nanmean(Xe[:, j]):9.3f})  contribution {contrib[j]:+.3f}")
    print()

print("######## LOWEST predicted scores ########")
for i in order_idx[:3]: explain(i)
print("######## HIGHEST predicted scores ########")
for i in order_idx[::-1][:3]: explain(i)

## 10 · Results log
Every row is computed from the cells above (nothing is typed in by hand).

In [ ]:
log = pd.DataFrame([
    dict(experiment="constant mean (label std)", cv_rmse=const_rmse, cv_pearson=np.nan),
    dict(experiment=f"best single: {best['name']}", cv_rmse=float(best.rmse_overall), cv_pearson=float(best.pearson_overall)),
    dict(experiment=f"final ensemble ({chosen}) of {cands}", cv_rmse=cv_ens_rmse, cv_pearson=cv_ens_pearson),
])
display(log.round(4))
log.to_csv(OUT / "results_log.csv", index=False)
print(f"TRAINING RMSE of the final ensemble: {train_rmse:.4f}")

# REPORT

## Approach
Convert each recording to text with a local Whisper model; describe the transcript and the timing of the speech with a small set of interpretable features and a few dense embeddings; regress the 0–5 score with simple models; combine the three best models with a non-negative linear stack; clip to [0, 5]. Everything runs locally on the Kaggle GPU; no external datasets are used.

## Preprocessing
Resample to 16 kHz mono → trim leading/trailing silence (internal pauses kept) → normalise loudness → cache. Files that cannot be read yield an empty transcript instead of stopping the run.

## Pipeline architecture
```
 .wav ──► preprocess (16 kHz mono, trim, loudness) ──► Whisper (local) ──► transcript + word timestamps + confidences
                     │                                                         │
                     │                                   ┌─────────────────────┼───────────────────────────┐
                     ▼                                   ▼                     ▼                           ▼
              WavLM embedding                  text statistics        GPT-2 perplexity        fluency (rate, pauses, ASR conf.)
              (mean+std over time)             (+ LanguageTool*)      sentence/RoBERTa embeddings   (+ LLM rubric score*)
                     │                                   │                     │                           │
                     └───────────── Ridge / SVR on embeddings ─────── Ridge / LightGBM on engineered features
                                                         │
                                  repeated stratified 5-fold CV → out-of-fold predictions
                                                         │
                                 top-3 models → non-negative linear stack → clip [0, 5] → submission.csv
                                                  (* optional)
```

## Evaluation results
The numbers are produced by the cells above and are not repeated here by hand: see the **ablation table** (section 6), the **final model summary** (section 8: training RMSE, CV RMSE, CV Pearson, constant baseline) and `results_log.csv`.

## Limitations
* **Whisper cleans speech.** It often omits fillers/false starts and may silently correct grammar, so the transcript can look better than what was said. Errors also depend on accent/audio quality, i.e. ASR errors are mixed with grammar errors.
* **Small data.** ~770 labelled files: fold-to-fold noise is large (see fold std); small RMSE differences between models are not reliable.
* **Label noise / subjectivity.** MOS-style grammar ratings are subjective; there is a ceiling on achievable accuracy.
* **Regression to the mean.** Models under-predict high scores and over-predict low ones (see per-score error plot); RMSE is dominated by mid-range files.
* **Pause features are approximate.** Whisper word timestamps are not forced alignments.
* **Perplexity is not grammar.** It also reflects topic, vocabulary rarity and punctuation chosen by Whisper.
* **WavLM embeddings** carry sound/pronunciation information, only indirectly related to grammar.
* **LanguageTool / LLM features are optional** and may be missing in a given run; check which groups the ablation actually contained.
* **Selection effect.** Picking the top-3 models on the same CV used to report them is slightly optimistic.
* **Training RMSE is optimistic** by construction; use CV RMSE for expectations.

## What I would try next
* Preserve disfluencies in the transcript (Whisper `initial_prompt`, a larger model, or a verbatim ASR) and compare.
* Fine-tune a small encoder (e.g. DeBERTa-v3) on transcripts with CV (more complex, so only if the simple pipeline plateaus).
* Add features from a parser (clause counts, tree depth) and a learner-error-aware grammar tool.
* Use forced alignment (e.g. CTC aligner) for more trustworthy pause measures.
* Tune LightGBM/SVR hyper-parameters with nested CV; try several seeds per model.
* Evaluate the optional LLM rubric score with a larger local model and a better prompt; calibrate it.
* Ordinal / classification-aware loss to reduce regression to the mean.

# Interview prep — every step in plain English

**1. Why transcribe first?** The label is about *grammar*, and grammar is easiest to judge on text. Whisper turns speech into text locally (data never leaves Kaggle). *Limits:* Whisper is trained to produce clean, well-formed text, so it may fix mistakes the speaker really made and drop "um"s; ASR errors for accented speech can look like grammar errors.

**2. Why preprocess the audio (16 kHz mono, trim, loudness)?** Models expect 16 kHz mono; trimming removes dead air at the edges (which would otherwise inflate "pause" measures); loudness normalisation removes recording-volume differences. *Limits:* trimming threshold is a heuristic; I deliberately keep internal pauses.

**3. Why cache?** Whisper is the slowest step. Running it once and saving text + timestamps makes every later experiment fast and reproducible.

**4. Text statistics (`hand_`).** Word count, vocabulary variety, sentence lengths, fillers, immediate repetitions, false starts, subordinating words. Each is a one-line idea (e.g. many fragments or repeats → weaker control; many "because/although/which" → attempts at complex sentences). *Limits:* they measure *attempted* structure, not whether it was correct; sentence boundaries come from Whisper's punctuation.

**5. GPT-2 perplexity (`lm_`).** A language model's average surprise per token. Natural, grammatical English is more predictable. *Limits:* also rises with rare topics/words; Whisper's punctuation affects it.

**6. LanguageTool (`lt_`, optional).** A rule-based grammar checker counting flagged errors per 100 words. Runs locally via Java; skipped if unavailable. *Limits:* tuned for written text; on clean Whisper output it may flag little.

**7. Fluency features (`flu_`).** Words per second, pause count/length (gaps ≥ 0.3 s between words), run length between pauses, and Whisper's own confidence. Hesitation and low intelligibility often go with weaker grammar control. *Limits:* word timestamps are approximate; speech rate also depends on personality and topic.

**8. Embeddings.** A sentence encoder and RoBERTa turn the transcript into a vector capturing meaning *and* style; WavLM does the same for the sound. A linear model on top lets the data decide what matters. *Limits:* high-dimensional vs ~770 rows → needs strong regularisation; individual dimensions are not interpretable; WavLM is only indirectly related to grammar.

**9. Why Ridge / SVR / LightGBM?** Ridge = linear + L2 penalty, the safe default for embeddings. SVR = smooth non-linear alternative. LightGBM = shallow trees for small tabular feature sets, handles missing values and thresholds. All are simple, fast and hard to over-tune with this little data. *Limits:* trees can over-fit small data, hence shallow depth and regularisation.

**10. Cross-validation (repeated, stratified, seeds fixed).** The model is trained on 4/5 of the data and scored on the unseen 1/5, five times, repeated with different splits. Stratifying on the rounded label keeps every fold's score mix similar. Imputers, scalers and the ridge penalty are fitted *inside* each training part so validation rows never influence them. *Limits:* with ~770 rows the estimate has noticeable noise (see fold std).

**11. Constant baseline.** Predicting the mean score gives RMSE ≈ label std. If a model is not clearly below that, it learned nothing.

**12. Metrics.** RMSE = typical size of the error in score points (penalises big misses). Pearson = whether the ranking/trend is right regardless of scale. The leaderboard uses both.

**13. Ablation table.** Each feature group and model evaluated separately, and engineered groups added cumulatively, so I can say what actually helped rather than guess.

**14. Ensemble.** The three best models make different mistakes; a non-negative weighted combination (weights learned on out-of-fold predictions) usually beats any one. A second-level CV checks that the weights themselves are not over-fitted. *Limits:* gains are often small; top-3 selection is slightly optimistic.

**15. Clipping to [0, 5].** Scores outside the valid range can only be wrong, so clipping can only help.

**16. Training RMSE vs CV RMSE.** Training RMSE is measured on data the model has seen (optimistic). CV RMSE is measured on unseen data and is the number to trust. A big gap = over-fitting.

**17. Submission check.** The writer verifies column names, order, row count, no NaN, and values in [0, 5] against `sample_submission.csv`.

**18. Rules compliance.** No external datasets; pretrained models used only as local feature extractors; no data sent to any API or third party (LanguageTool runs a local server); test labels never used; all fitting inside CV folds.

**Questions to be ready for:** *Why not fine-tune a transformer?* (small data, complexity, explainability.) *Why not trust training RMSE?* (it is optimistic.) *What if Whisper hides the errors?* (main limitation; next step: verbatim ASR / disfluency prompt.) *How do you know features are not leaking?* (everything fitted in-fold; test labels never used.)